In [ ]:
pip install --upgrade openai

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
pip install beautifulsoup4 lxml

In [ ]:
!pip install playwright
import asyncio
from playwright.async_api import async_playwright

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 21.4 MB/s eta 0:00:00


In [ ]:
#!pip install playwright
!playwright install chromium
!playwright install-deps

177 MiB [] 0% 0.0s177 MiB [] 0% 181.9s177 MiB [] 0% 204.3s177 MiB [] 0% 657.8s177 MiB [] 0% 555.6s177 MiB [] 0% 745.9s177 MiB [] 0% 661.1s177 MiB [] 0% 598.9s177 MiB [] 0% 553.7s177 MiB [] 0% 517.3s177 MiB [] 0% 489.3s177 MiB [] 0% 474.6s177 MiB [] 0% 453.9s177 MiB [] 0% 436.3s177 MiB [] 0% 421.3s177 MiB [] 0% 408.2s177 MiB [] 0% 394.0s177 MiB [] 0% 382.1s177 MiB [] 0% 362.9s177 MiB [] 0% 348.4s177 MiB [] 0% 321.8s177 MiB [] 0% 305.3s177 MiB [] 0% 288.8s177 MiB [] 0% 274.9s177 MiB [] 0% 263.3s177 MiB [] 0% 246.5s177 MiB [] 0% 237.6s177 MiB [] 0% 224.7s177 MiB [] 0% 217.9s177 MiB [] 0% 203.4s177 MiB [] 0% 190.7s177 MiB [] 0% 179.6s177 MiB [] 0% 170.1s177 MiB [] 0% 159.3s177 MiB [] 0% 151.7s177 MiB [] 0% 139.8s177 MiB [] 0% 129.8s177 MiB [] 0% 122.7s177 MiB [] 0% 115.5s177 MiB [] 0% 109.1s177 MiB [] 0% 102.8s177 MiB [] 1% 98.0s177 MiB [] 1% 91.8s177 MiB [] 1% 86.2s177 MiB [] 1% 83.9s177 MiB [] 1% 74.4s177 MiB [] 1% 70.3s177 MiB [] 1% 65.4s177 MiB [] 1% 61.2s177 MiB [] 1% 58.2s177 MiB [] 

In [ ]:
import sys
import pandas as pd
import asyncio
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError
from bs4 import BeautifulSoup
import re
import os
import csv
import random
from pathlib import Path

In [ ]:
!pip install playwright-stealth

## Walmart Feminine Wipes ##

## Preparing for parsing Walmart HTML files ##

In [ ]:
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd
import re
from urllib.parse import urljoin

BASE = "https://www.walmart.com"

html_files = [
    Path("/content/drive/My Drive/Personal Care Project/Web Scraping_Wipes/Walmart Feminine Wipes/feminine care wipes - Walmart.com one.html"),
    Path("/content/drive/My Drive/Personal Care Project/Web Scraping_Wipes/Walmart Feminine Wipes/feminine care wipes - Walmart.com two.html"),
    Path("/content/drive/My Drive/Personal Care Project/Web Scraping_Wipes/Walmart Feminine Wipes/feminine care wipes - Walmart.com three.html"),
    Path("/content/drive/My Drive/Personal Care Project/Web Scraping_Wipes/Walmart Feminine Wipes/feminine care wipes - Walmart.com four.html"),
    Path("/content/drive/My Drive/Personal Care Project/Web Scraping_Wipes/Walmart Feminine Wipes/feminine care wipes - Walmart.com five.html"),
    Path("/content/drive/My Drive/Personal Care Project/Web Scraping_Wipes/Walmart Feminine Wipes/feminine care wipes - Walmart.com six.html"),
    Path("/content/drive/My Drive/Personal Care Project/Web Scraping_Wipes/Walmart Feminine Wipes/feminine care wipes - Walmart.com seven.html"),
    Path("/content/drive/My Drive/Personal Care Project/Web Scraping_Wipes/Walmart Feminine Wipes/feminine care wipes - Walmart.com eight.html"),
    Path("/content/drive/My Drive/Personal Care Project/Web Scraping_Wipes/Walmart Feminine Wipes/feminine care wipes - Walmart.com nine.html"),
]

def clean(x):
    return re.sub(r"\s+", " ", x or "").strip()

def first_text(node, selectors):
    for sel in selectors:
        el = node.select_one(sel)
        if el:
            txt = clean(el.get_text(" ", strip=True))
            if txt:
                return txt
    return ""

def first_attr(node, selectors, attr):
    for sel in selectors:
        el = node.select_one(sel)
        if el and el.has_attr(attr):
            val = clean(el.get(attr))
            if val:
                return val
    return ""

## Parse Walmart HTML files and extract detailed product information ##

In [ ]:
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd
import re
from urllib.parse import urljoin, urlparse, parse_qs, unquote
import csv

# Helper functions (copied from hjnvFdvsjxT- to ensure self-contained parsing)
# BASE variable is not strictly needed for local file parsing but included for consistency.
BASE = "https://www.walmart.com"

# Define FOLDER for Walmart files, similar to the Target script
FOLDER = Path("/content/drive/My Drive/Personal Care Project/Web Scraping_Wipes/Walmart Feminine Wipes")

def clean(text: str) -> str:
    return re.sub(r"\s+", " ", (text or "")).strip()

def extract_product_id(href: str) -> str:
    """
    Pulls a stable product identifier out of a Walmart product link so we can
    de-duplicate. Handles both direct /ip/.../<id> links and the wrapped
    /sp/track...?rd=<url-encoded redirect> sponsored-listing links.
    """
    if not href:
        return ""

    href = unquote(href)

    # Sponsored links wrap the real destination in the "rd" query param.
    if "/sp/track" in href:
        qs = parse_qs(urlparse(href).query)
        rd = qs.get("rd", [""])[0]
        if rd:
            href = unquote(rd)

    m = re.search(r"/ip/[^/]+/(\d+)", href)
    if m:
        return m.group(1)

    # Fallback: use the whole path minus query string.
    return urlparse(href).path

def parse_item(item_soup) -> dict | None:
    """
    Extracts ProductTitle, UnitPrice, PricePer100ct, Rating, RatingCount
    from a single Walmart product-tile <div>.
    """

    # --- Product link / id (used for de-duplication) ---
    link_el = item_soup.select_one("a[link-identifier], a[href*='/ip/']")
    href = link_el.get("href", "") if link_el else ""
    product_id = extract_product_id(href)
    if not product_id:
        return None
    product_web_link = urljoin(BASE, href)

    # --- Title ---
    title = ""
    title_el = item_soup.select_one(
        "span[data-automation-id='product-title'], "
        "a[link-identifier='linkProductTitle'] span, "
        "[data-testid='product-title']"
    )
    if title_el:
        title = title_el.get_text(" ", strip=True)
    if not title:
        # Fallback: the h3/h2 heading Walmart renders under the tile.
        heading_el = item_soup.select_one("span.w_V_DM, h3, h2")
        if heading_el:
            title = heading_el.get_text(" ", strip=True)
    title = clean(title)
    if not title:
        return None

    # --- Price block text (keep separate from the rest of the tile so we
    #     don't accidentally pull numbers from unrelated badges) ---
    price_block = item_soup.select_one(
        "div[data-automation-id='product-price'], "
        "[data-testid='list-view-price'], "
        "[data-testid='price-container']"
    )
    price_text = price_block.get_text(" ", True) if price_block else ""
    if not price_text:
        # Fall back to the whole tile if a dedicated price block isn't found.
        price_text = item_soup.get_text(" ", True)

    # Unit price / current price, e.g. "current price $3.78" or
    # "current price Now $5.88, Was $12.58"
    unit_price = ""
    m = re.search(
        r"current price\s+(?:Now\s+)?\$?([\d,]+\.\d{2})", price_text, re.I
    )
    if m:
        unit_price = f"${m.group(1)}"
    else:
        # Fallback: first plain "$X.XX" that isn't part of an options range.
        m = re.search(r"\$([\d,]+\.\d{2})", price_text)
        if m:
            unit_price = f"${m.group(1)}"

    # Price per 100 ct, e.g. "$11.81/100 ct" — leave blank if not shown.
    price_per_100ct = ""
    m = re.search(r"\$[\d,]+\.\d{2}\s*/\s*100\s*ct", price_text, re.I)
    if m:
        price_per_100ct = clean(m.group(0))

    # --- Rating & rating count ---
    rating = ""
    rating_count = ""

    # Pattern for rating score, more flexible (e.g., "4.7 out of 5 stars" or "4.7 stars.")
    rating_score_pattern = re.compile(r"(\d+(?:\.\d+)?)\s*(?:out of 5)?\s*stars?(\.)?", re.I)
    # Pattern for review count, allowing for singular 'review' or plural 'reviews' and comma in number
    reviews_count_pattern = re.compile(r"([\d,]+)\s*reviews?", re.I)

    # 1. Prioritize aria-label attributes for a comprehensive rating/count string
    for el in item_soup.find_all(attrs={"aria-label": True}):
        label = el.get("aria-label", "")
        # Only process labels that clearly contain rating/review info to avoid false positives
        if ("star" in label.lower() and ("rating" in label.lower() or "review" in label.lower() or "out of 5" in label.lower())):
            if not rating: # Only set if rating is not yet found
                m_score = rating_score_pattern.search(label)
                if m_score:
                    rating = m_score.group(1)

            if not rating_count: # Only set if count is not yet found
                m_count = reviews_count_pattern.search(label)
                if m_count:
                    rating_count = m_count.group(1).replace(",", "")

            # If both are found, we can stop searching aria-labels
            if rating and rating_count:
                break

    # 2. Fallback: look in dedicated reviews/ratings container if rating/count still missing
    # This block will only execute if 'rating' or 'rating_count' are still empty after aria-label check
    if not rating or not rating_count:
        reviews_el = item_soup.select_one(
            "[data-testid='product-reviews'], "
            "[data-testid='reviews-count'], "
            "span[data-automation-id='product-stars'],"
            "div.w_rB, " # Common container for rating and review count (e.g. <div class="w_rB">...</div>)
            "span.e_iE, " # Another common pattern for rating text (e.g. <span class="e_iE">4.7 out of 5 stars</span>)
            "div.ratings-summary span, " # Yet another pattern seen in Walmart HTML
            "div[aria-label][role='img']" # A common pattern for star ratings where aria-label holds the full text
        )
        if reviews_el:
            search_text = reviews_el.get_text(" ", strip=True) + " " + (reviews_el.get("aria-label", "") or "")

            if not rating: # Only try to find rating if it's still missing
                m_score = rating_score_pattern.search(search_text)
                if m_score:
                    rating = m_score.group(1)

            if not rating_count: # Only try to find count if it's still missing
                m_count = reviews_count_pattern.search(search_text)
                if m_count:
                    rating_count = m_count.group(1).replace(",", "")

    # 3. Final comprehensive fallback: search the entire item's text if rating or count are still not found
    # This captures cases where rating/count might be in less structured parts of the HTML.
    if not rating or not rating_count:
        full_item_text = item_soup.get_text(" ", strip=True) + " " + (item_soup.get("aria-label", "") or "")
        if not rating:
            m_score_full = rating_score_pattern.search(full_item_text)
            if m_score_full:
                rating = m_score_full.group(1)
        if not rating_count:
            m_count_full = reviews_count_pattern.search(full_item_text)
            if m_count_full:
                rating_count = m_count_full.group(1).replace(",", "")

    return {
        "ProductWebLink": product_web_link,
        "ProductTitle": title,
        "UnitPrice": unit_price,
        "PricePer100ct": price_per_100ct,
        "Rating": rating,
        "RatingCount": rating_count,
        "ProductId": product_id,
    }

# List of HTML files to process, now using the FOLDER variable
html_files = [
    FOLDER / "feminine care wipes - Walmart.com one.html",
    FOLDER / "feminine care wipes - Walmart.com two.html",
    FOLDER / "feminine care wipes - Walmart.com three.html",
    FOLDER / "feminine care wipes - Walmart.com four.html",
    FOLDER / "feminine care wipes - Walmart.com five.html",
    FOLDER / "feminine care wipes - Walmart.com six.html",
    FOLDER / "feminine care wipes - Walmart.com seven.html",
    FOLDER / "feminine care wipes - Walmart.com eight.html",
    FOLDER / "feminine care wipes - Walmart.com nine.html",
]

all_results = []
seen_product_ids = set() # To handle potential duplicates across files and within pages

for file_path in html_files:
    print(f"Processing file: {file_path.name}")
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            soup = BeautifulSoup(f, "html.parser") # Use "html.parser" for robustness

        # Select all product tiles using a robust selector identified in previous steps
        items = soup.select("div[data-item-id]")
        if not items:
            # Fallback: derive tiles from product-title spans' ancestors.
            items = [
                t.find_parent("div")
                for t in soup.select(
                    "span[data-automation-id='product-title']"
                )
            ]
            items = [i for i in items if i is not None]

        page_new_count = 0
        for item_soup in items:
            data = parse_item(item_soup)
            if data and data["ProductId"] not in seen_product_ids:
                all_results.append(data)
                seen_product_ids.add(data["ProductId"])
                page_new_count += 1
        print(f"  Found {len(items)} tiles, {page_new_count} new unique items from {file_path.name}.")
    except FileNotFoundError:
        print(f"  Error: File not found - {file_path.name}. Skipping.")
    except Exception as e:
        print(f"  Error processing {file_path.name}: {e}. Skipping.")

if all_results:
    df = pd.DataFrame(all_results)
    # Drop the internal ProductId helper field from the final CSV as it's not requested in the output.
    df = df.drop(columns=["ProductId"])

    # Output CSV path, now using FOLDER for consistency
    output_csv_path = FOLDER / "walmart_feminine_wipes_extracted.csv"
    # Add ProductWebLink to the fieldnames for the CSV output
    fieldnames = [
        "ProductWebLink",
        "ProductTitle",
        "UnitPrice",
        "PricePer100ct",
        "Rating",
        "RatingCount",
    ]
    with open(output_csv_path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in all_results:
            writer.writerow({k: row.get(k, "") for k in fieldnames})

    print(f"Done! Saved {len(df)} unique items to {output_csv_path}")
else:
    print("No results collected from any files; nothing written.")

Processing file: feminine care wipes - Walmart.com one.html
  Found 44 tiles, 44 new unique items from feminine care wipes - Walmart.com one.html.
Processing file: feminine care wipes - Walmart.com two.html
  Found 40 tiles, 40 new unique items from feminine care wipes - Walmart.com two.html.
Processing file: feminine care wipes - Walmart.com three.html
  Found 40 tiles, 40 new unique items from feminine care wipes - Walmart.com three.html.
Processing file: feminine care wipes - Walmart.com four.html
  Found 40 tiles, 40 new unique items from feminine care wipes - Walmart.com four.html.
Processing file: feminine care wipes - Walmart.com five.html
  Found 40 tiles, 29 new unique items from feminine care wipes - Walmart.com five.html.
Processing file: feminine care wipes - Walmart.com six.html
  Found 40 tiles, 39 new unique items from feminine care wipes - Walmart.com six.html.
Processing file: feminine care wipes - Walmart.com seven.html
  Found 40 tiles, 37 new unique items from femin

## Target Feminine Wipes ##

In [ ]:

from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd
import re
from urllib.parse import urljoin

BASE = "https://www.target.com"

FOLDER = Path("/content/drive/My Drive/Personal Care Project/Web Scraping_Wipes/Target Feminine Wipes")

html_files = [
    FOLDER / "_feminine wipes_ _ Target one.html",
    FOLDER / "_feminine wipes_ _ Page 2 _ Target two.html",
    FOLDER / "_feminine wipes_ _ Page 3 _ Target three.html",
    FOLDER / "_feminine wipes_ _ Page 4 _ Target four.html",
    FOLDER / "_feminine wipes_ _ Page 5 _ Target five.html",
    FOLDER / "_feminine wipes_ _ Page 6 _ Target six.html",
    FOLDER / "_feminine wipes_ _ Page 7 _ Target seven.html",
    FOLDER / "_feminine wipes_ _ Page 8 _ Target eight.html",
    FOLDER / "_feminine wipes_ _ Page 9 _ Target nine.html",
]

def clean(x):
    return re.sub(r"\s+", " ", x or "").strip()

def first_text(node, selectors):
    for sel in selectors:
        el = node.select_one(sel)
        if el:
            txt = clean(el.get_text(" ", strip=True))
            if txt:
                return txt
    return ""

def first_attr(node, selectors, attr):
    for sel in selectors:
        el = node.select_one(sel)
        if el and el.has_attr(attr):
            val = clean(el.get(attr))
            if val:
                return val
    return ""

def extract_rating_count(text):
    if not text:
        return "", ""
    t = clean(text)
    rating = ""
    count = ""
    m_rating = re.search(r"([0-5](?:\.\d)?)\s*(?:out of 5)?", t, re.I)
    if m_rating:
        rating = m_rating.group(1)
    m_count = re.search(r"([\d,]+)\s*(?:ratings?|reviews?)", t, re.I)
    if m_count:
        count = m_count.group(1).replace(",", "")
    if not count:
        nums = re.findall(r"[\d,]+", t)
        if nums:
            count = nums[-1].replace(",", "")
    return rating, count

def parse_one_page(soup):
    rows = []
    # Target product tiles commonly use data-test attributes
    product_tiles = soup.select(
        '[data-test="product-card"], [data-test="@web/site-top-of-funnel/ProductCardWrapper"], div[data-test*="product-card"]'
    )
    if not product_tiles:
        # fallback broader selector
        product_tiles = soup.select('a[href*="/p/"]')

    seen = set()

    for tile in product_tiles:
        html_key = str(tile)[:500]
        if html_key in seen:
            continue
        seen.add(html_key)

        link = first_attr(tile, [
            'a[href*="/p/"]',
            "a[href]"
        ], "href")

        title = first_attr(tile, [
            'a[data-test="product-title"]',
            'a[href*="/p/"]'
        ], "aria-label")

        if not title:
            title = first_text(tile, [
                'a[data-test="product-title"]',
                '[data-test="product-title"]',
                "h3",
                'a[href*="/p/"]'
            ])

        price = first_text(tile, [
            '[data-test="product-price"]',
            'span[data-test="current-price"]',
            'span[class*="price"]'
        ])
        m = re.search(r"\$?\s*([\d]+(?:\.\d{2})?)", price)
        price = m.group(1) if m else ""

        price_per_100 = ""
        all_text = clean(tile.get_text(" ", strip=True))
        m_100 = re.search(r"(\$[\d]+(?:\.\d{1,2})?/\s*100\s*ct)\b", all_text, re.I)
        if m_100:
            price_per_100 = m_100.group(1).replace(" ", "")

        rating_text = first_attr(tile, [
            '[aria-label*="stars"]',
            '[aria-label*="rating"]',
            'a[data-test="ratings"]',
            'span[data-test="ratings"]'
        ], "aria-label")

        if not rating_text:
            rating_text = first_text(tile, [
                '[data-test="ratings"]',
                'a[href*="reviews"]',
                'span[class*="rating"]'
            ])

        rating, rating_count = extract_rating_count(rating_text)

        if link or title:
            rows.append({
                "product_web_link": urljoin(BASE, link) if link else "",
                "product_title": title,
                "product_unit_price": price,
                "price_per_100_ct": price_per_100,
                "rating": rating,
                "rating_counts": rating_count,
                "source_file": None,
            })
    return rows

all_rows = []
for fp in html_files:
    if not fp.exists():
        print(f"WARNING: file not found -> {fp}")
        continue
    with open(fp, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "lxml")
    page_rows = parse_one_page(soup)
    for r in page_rows:
        r["source_file"] = fp.name
    all_rows.extend(page_rows)
    print(f"Parsed {len(page_rows)} products from {fp.name}")

df = pd.DataFrame(all_rows).drop_duplicates(subset=["product_web_link", "product_title"])
output_path = FOLDER / "target_feminine_wipes_extracted.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(df.head(10))
print(f"Total rows: {len(df)}")
print(f"Saved to: {output_path}")


Parsed 30 products from _feminine wipes_ _ Target one.html
Parsed 30 products from _feminine wipes_ _ Page 2 _ Target two.html
Parsed 30 products from _feminine wipes_ _ Page 3 _ Target three.html
Parsed 24 products from _feminine wipes_ _ Page 4 _ Target four.html
Parsed 24 products from _feminine wipes_ _ Page 5 _ Target five.html
Parsed 24 products from _feminine wipes_ _ Page 6 _ Target six.html
Parsed 24 products from _feminine wipes_ _ Page 7 _ Target seven.html
Parsed 24 products from _feminine wipes_ _ Page 8 _ Target eight.html
Parsed 21 products from _feminine wipes_ _ Page 9 _ Target nine.html
                                     product_web_link product_title  \
0   https://www.target.com/p/waterwipes-feminine-w...                 
1   https://www.target.com/p/vagisil-maximum-stren...                 
2   https://www.target.com/p/summer-s-eve-blissful...                 
3   https://www.target.com/p/feminine-wipes-unscen...                 
4   https://www.target.com/p/summ

In [ ]:
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd
import re
from urllib.parse import urljoin
import csv

# BASE URL for Target
BASE = "https://www.target.com"

# Folder where Target HTML files are stored
FOLDER = Path("/content/drive/My Drive/Personal Care Project/Web Scraping_Wipes/Target Feminine Wipes")

# List of Target HTML files to process
html_files = [
    FOLDER / "_feminine wipes_ _ Target one.html",
    FOLDER / "_feminine wipes_ _ Page 2 _ Target two.html",
    FOLDER / "_feminine wipes_ _ Page 3 _ Target three.html",
    FOLDER / "_feminine wipes_ _ Page 4 _ Target four.html",
    FOLDER / "_feminine wipes_ _ Page 5 _ Target five.html",
    FOLDER / "_feminine wipes_ _ Page 6 _ Target six.html",
    FOLDER / "_feminine wipes_ _ Page 7 _ Target seven.html",
    FOLDER / "_feminine wipes_ _ Page 8 _ Target eight.html",
    FOLDER / "_feminine wipes_ _ Page 9 _ Target nine.html",
]

def clean(x):
    """Cleans up whitespace from a string."""
    return re.sub(r"\s+", " ", x or "").strip()

def first_text(node, selectors):
    """Finds the first matching element by selectors and returns its cleaned text."""
    for sel in selectors:
        el = node.select_one(sel)
        if el:
            txt = clean(el.get_text(" ", strip=True))
            if txt:
                return txt
    return ""

def first_attr(node, selectors, attr):
    """Finds the first matching element by selectors and returns the value of its attribute."""
    for sel in selectors:
        el = node.select_one(sel)
        if el and el.has_attr(attr):
            val = clean(el.get(attr))
            if val:
                return val
    return ""

def extract_rating_count(text):
    """Extracts rating score and count from a text string."""
    if not text:
        return "", ""
    t = clean(text)
    rating = ""
    count = ""
    # Pattern for rating: e.g., "4.5 out of 5", "4.5"
    m_rating = re.search(r"([0-5](?:\.\d)?)\s*(?:out of 5)?", t, re.I)
    if m_rating:
        rating = m_rating.group(1)
    # Pattern for count: e.g., "1,234 ratings", "500 reviews"
    m_count = re.search(r"([\d,]+)\s*(?:ratings?|reviews?)", t, re.I)
    if m_count:
        count = m_count.group(1).replace(",", "")
    if not count:
        # Fallback: if no specific "ratings/reviews" found, try to find any numbers
        nums = re.findall(r"[\d,]+", t)
        if nums:
            count = nums[-1].replace(",", "") # Assume the last number is the count

    return rating, count

def parse_target_item(tile_soup) -> dict | None:
    """
    Extracts ProductWebLink, ProductTitle, UnitPrice, PricePer100ct, Rating, RatingCount
    from a single Target product tile.
    """
    link = first_attr(tile_soup, [
        'a[href*="/p/"]',
        "a[href]"
    ], "href")

    # 1. Try to get title from alt attribute of an image first
    title = first_attr(tile_soup, [
        'img[alt]', # General image alt
        '[data-test="@web/ProductCard/ProductCardImage/primary"] img', # More specific image alt
        '.styles_productCardImageWrapper__ZrSgM img' # Another specific image alt
    ], "alt")

    if not title:
        # 2. If not found, fallback to aria-label on product title elements/links
        title = first_attr(tile_soup, [
            'a[data-test="product-title"]', # Specific for Target titles
            'a[href*="/p/"]' # Main product link
        ], "aria-label")

    if not title:
        # 3. If still not found, try to get title from text content of various elements
        title = first_text(tile_soup, [
            'a[data-test="product-title"]', # Text content of specific title link
            '[data-test="product-title"]', # Text content of specific title div/span
            'div.styles_name__N-t1U', # Common class for product name
            'span[data-test="product-title"]', # Common data-test for span
            'a[href*="/p/"]', # Text of the product link itself
            "h3", # General heading
            "h2"  # General heading
        ])

    title = clean(title) # Ensure title is cleaned after all attempts

    # Extract price
    price = first_text(tile_soup, [
        '[data-test="product-price"]',
        'span[data-test="current-price"]',
        'span[class*="price"]'
    ])
    m = re.search(r"\$?+\s*([\d]+(?:\.\d{2})?)", price)
    unit_price = m.group(1) if m else ""

    # Extract price per 100 ct
    price_per_100ct = ""
    all_text = clean(tile_soup.get_text(" ", strip=True))
    m_100 = re.search(r"(\$[\d]+(?:\.\d{1,2})?)/\s*100\s*ct\b", all_text, re.I)
    if m_100:
        price_per_100ct = m_100.group(1).replace(" ", "")

    # Extract rating and rating count
    rating_text = first_attr(tile_soup, [
        '[aria-label*="stars"]',
        '[aria-label*="rating"]',
        'a[data-test="ratings"]',
        'span[data-test="ratings"]'
    ], "aria-label")

    if not rating_text:
        rating_text = first_text(tile_soup, [
            '[data-test="ratings"]',
            'a[href*="reviews"]',
            'span[class*="rating"]'
        ])

    rating, rating_count = extract_rating_count(rating_text)

    # Only return an item if it has at least a link or a title
    if link or title:
        return {
            "product_web_link": urljoin(BASE, link) if link else "",
            "product_title": title,
            "product_unit_price": unit_price,
            "price_per_100_ct": price_per_100ct,
            "rating": rating,
            "rating_counts": rating_count,
        }
    return None

all_results = []

for file_path in html_files:
    print(f"Processing file: {file_path.name}")
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            soup = BeautifulSoup(f, "lxml") # Using lxml for speed if available, otherwise html.parser

        # Target product tiles selectors, adapted from Jm6_oxKt-t-b
        product_tiles = soup.select(
            '[data-test="product-card"], [data-test="@web/site-top-of-funnel/ProductCardWrapper"], div[data-test*="product-card"]'
        )
        if not product_tiles:
            # fallback broader selector as seen in Jm6_oxKt-t-b
            product_tiles = soup.select('a[href*="/p/"]')
            # If still no specific tiles, consider each 'a' tag with a product link as a "tile" for simpler parsing
            # This might be less robust but catches products that are just direct links
            if not product_tiles:
                product_tiles = soup.select('div.ProductCardWrapper') # Another common wrapper

        page_new_count = 0
        for tile in product_tiles:
            data = parse_target_item(tile) # Call the new Target-specific parser
            if data:
                data["source_file"] = file_path.name # Add source file info
                all_results.append(data)
                page_new_count += 1
        print(f"  Parsed {page_new_count} products from {file_path.name}.")
    except FileNotFoundError:
        print(f"  Error: File not found - {file_path.name}. Skipping.")
    except Exception as e:
        print(f"  Error processing {file_path.name}: {e}. Skipping.")

if all_results:
    # Create DataFrame and drop duplicates based on product_web_link and product_title
    df = pd.DataFrame(all_results).drop_duplicates(subset=["product_web_link", "product_title"]) # Ensure the columns exist after parsing

    output_csv_path = FOLDER / "target_feminine_wipes_extracted.csv" # Output to the same folder as input files

    # Define fieldnames explicitly to control CSV column order
    fieldnames = [
        "product_web_link",
        "product_title",
        "product_unit_price",
        "price_per_100_ct",
        "rating",
        "rating_counts",
        "source_file",
    ]
    with open(output_csv_path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(df.to_dict('records')) # Write all unique rows from the DataFrame

    print(f"Done! Saved {len(df)} unique items to {output_csv_path}")
    print(df.head(10)) # Display head of the final DataFrame
else:
    print("No results collected from any files; nothing written.")

Processing file: _feminine wipes_ _ Target one.html
  Parsed 30 products from _feminine wipes_ _ Target one.html.
Processing file: _feminine wipes_ _ Page 2 _ Target two.html
  Parsed 30 products from _feminine wipes_ _ Page 2 _ Target two.html.
Processing file: _feminine wipes_ _ Page 3 _ Target three.html
  Parsed 30 products from _feminine wipes_ _ Page 3 _ Target three.html.
Processing file: _feminine wipes_ _ Page 4 _ Target four.html
  Parsed 24 products from _feminine wipes_ _ Page 4 _ Target four.html.
Processing file: _feminine wipes_ _ Page 5 _ Target five.html
  Parsed 24 products from _feminine wipes_ _ Page 5 _ Target five.html.
Processing file: _feminine wipes_ _ Page 6 _ Target six.html
  Parsed 24 products from _feminine wipes_ _ Page 6 _ Target six.html.
Processing file: _feminine wipes_ _ Page 7 _ Target seven.html
  Parsed 24 products from _feminine wipes_ _ Page 7 _ Target seven.html.
Processing file: _feminine wipes_ _ Page 8 _ Target eight.html
  Parsed 24 product

## Walgreens Feminine Wipes ##

In [ ]:
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd
import re
from urllib.parse import urljoin
import os

BASE = "https://www.walgreens.com"

FOLDER = Path("/content/drive/My Drive/Personal Care Project/Web Scraping_Wipes/Walgreens Feminine Wipes")

# Dynamically get file paths from the directory to avoid manual typos
html_files = []
if FOLDER.exists() and FOLDER.is_dir():
    for filename in os.listdir(FOLDER):
        # Corrected: changed 'feminine wipe' to 'feminine wipes' to match actual filenames
        if filename.startswith("feminine wipes _ Walgreens") and filename.endswith(".html"):
            html_files.append(FOLDER / filename)
    html_files.sort() # Sort to ensure consistent order
else:
    print(f"Error: Directory not found at {FOLDER}")

def clean(x):
    return re.sub(r"\s+", " ", x or "").strip()

def first_text(node, selectors):
    for sel in selectors:
        el = node.select_one(sel)
        if el:
            txt = clean(el.get_text(" ", strip=True))
            if txt:
                return txt
    return ""

def first_attr(node, selectors, attr):
    for sel in selectors:
        el = node.select_one(sel)
        if el and el.has_attr(attr):
            val = clean(el.get(attr))
            if val:
                return val
    return ""

def extract_rating_count(text):
    if not text:
        return "", ""
    t = clean(text)
    rating = ""
    count = ""
    m_rating = re.search(r"([0-5](?:\.\d)?)\s*(?:out of 5)?", t, re.I)
    if m_rating:
        rating = m_rating.group(1)
    m_count = re.search(r"([\d,]+)\s*(?:ratings?|reviews?)", t, re.I)
    if m_count:
        count = m_count.group(1).replace(",", "")
    if not count:
        nums = re.findall(r"[\d,]+", t)
        if nums:
            count = nums[-1].replace(",", "")
    return rating, count

def parse_walgreens_item(tile_soup):

    link = first_attr(tile_soup, [
        'a[data-qa="product_link"]',
        'a[href*="/store/"]'
    ], "href")

    title = first_text(tile_soup, [
        'h1[itemprop="name"]',
        '#product-details h1.product-details__product-name',
        'h2.product-card__title',
        'a.product-card__title',
        '.wag-product-card__product-name',
        'p.wag-product-card__product-name',
        'span[data-qa="product_name"]',
        'a[data-automation="product-name"]',
        '.product-name-link',
        '.product-card__details__name',
        'h3.wag-product-card__product-name',
        'span[itemprop="name"]',
        'div[itemprop="name"]',
        'h2',
        'h3'
    ])
    if not title:
        title = first_attr(tile_soup, [
            'a.product-card__link',
            'a[href*="/store/"]',
            'img.product-card__image[alt]'
        ], "aria-label")
    if not title:
        title = first_attr(tile_soup, [
            'img.product-card__image'
        ], "alt")
    if not title:
        title = first_text(tile_soup, [
            'p.ProductTitle_ProductTitle__name__24G1w'
        ])

    price = first_text(tile_soup, [
        'div[itemprop="price"]',
        'span[itemprop="price"]',
        '#product-details span.wag-price-display__amount',
        '.wag-price-display__actual-value',
        'span[data-automation="product-price"]',
        '.product-card__price',
        '.product__price',
        'span[data-qa="current_price"]',
        'span.wag-price-display__amount',
        '.price',
        'span.current-price',
        'div.price-value',
        'strong.price',
        '#product-details .wag-price-display__actual-value'
    ])
    # Corrected regex to correctly extract price if present. Previously had '$?+' which is not valid.
    m = re.search(r"\$?\s*([\d,]+\.\d{2})", price)
    unit_price = m.group(1) if m else ""

    price_per_unit_measurement = "0"
    all_text = clean(tile_soup.get_text(" ", strip=True))

    m_unit_price = re.search(r"(\$[\d,]+(?:\.\d{1,2})?)\s*/\s*(?:oz|ea|ct|item|sheet)\b", all_text, re.I)
    if m_unit_price:
        price_per_unit_measurement = m_unit_price.group(0).replace(" ", "")

    rating_text = first_attr(tile_soup, [
        'div[itemprop="aggregateRating"][aria-label]',
        '.wag-rating-stars[aria-label]',
        '.wag-review-stars[aria-label]',
        '[data-qa="rating_stars_aria_label"]',
        'div.product-rating[aria-label]',
        'div.BVRRRatingSummaryLink, div.product-details__rating-summary'
    ], "aria-label")

    if not rating_text:
        rating_text = first_text(tile_soup, [
            'span[itemprop="ratingValue"]',
            'span[itemprop="ratingCount"]',
            '.wag-review-stars__average',
            '.wag-review-stars__reviews-count',
            'span.product-card__rating-score',
            '.wag-rating-stars__score',
            '.wag-review-stars__rating-summary',
            'span[data-qa="rating_value"]',
            'span[data-qa="reviews_count"]',
            '.wag-reviews-count__label',
            'div.review-total',
            'div.BVRRRatingSummaryLink, div.product-details__rating-summary'
        ])

    rating, rating_counts = extract_rating_count(rating_text)

    if link or title:
        return {
            "product_web_link": urljoin(BASE, link) if link else "",
            "product_title": title,
            "product_unit_price": unit_price,
            "price_per_100_ct": price_per_unit_measurement,
            "rating": rating,
            "rating_counts": rating_counts,
            "source_file": None,
        }
    return None

all_rows = []
for fp in html_files:
    print(f"Processing file: {fp.name}")
    if not fp.exists():
        print(f"WARNING: file not found -> {fp}")
        continue
    with open(fp, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "lxml")

    product_tiles = soup.select(
        'div.product-card-container, li.wag-product-card, div[data-shelf-item-id], [data-qa="product_tile"], .wag-product-card-wrapper, div[itemtype*="Product"], div[class*="product-details-page"], div#product-details-page'
    )
    if not product_tiles:
        product_tiles = [soup]

    page_new_count = 0
    seen_on_page = set()

    for tile in product_tiles:
        tile_key = str(tile)[:500]
        if tile_key in seen_on_page:
            continue
        seen_on_page.add(tile_key)

        data = parse_walgreens_item(tile)
        if data:
            data["source_file"] = fp.name
            all_rows.append(data)
            page_new_count += 1
    print(f"Parsed {page_new_count} products from {fp.name}")

df = pd.DataFrame(all_rows).drop_duplicates(subset=["product_web_link", "product_title"])
output_path = FOLDER / "walgreens_feminine_wipes_extracted.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(df.head(10))
print(f"Total rows: {len(df)}")
print(f"Saved to: {output_path}")

Processing file: feminine wipes _ Walgreens one.html
Parsed 1 products from feminine wipes _ Walgreens one.html
Processing file: feminine wipes _ Walgreens two.html
Parsed 1 products from feminine wipes _ Walgreens two.html
                                    product_web_link product_title  \
0  https://www.walgreens.com/store/c/cottonelle-s...                 
1  https://www.walgreens.com/store/c/always-pads-...                 

  product_unit_price price_per_100_ct rating rating_counts  \
0                            $0.25/ea                        
1                            $0.49/ea                        

                           source_file  
0  feminine wipes _ Walgreens one.html  
1  feminine wipes _ Walgreens two.html  
Total rows: 2
Saved to: /content/drive/My Drive/Personal Care Project/Web Scraping_Wipes/Walgreens Feminine Wipes/walgreens_feminine_wipes_extracted.csv


In [ ]:
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd
import re
from urllib.parse import urljoin
import csv

# BASE URL for Walgreens
BASE = "https://www.walgreens.com"

# Folder where Walgreens HTML files are stored
FOLDER = Path("/content/drive/My Drive/Personal Care Project/Web Scraping_Wipes/Walgreens Feminine Wipes")

# List of Walgreens HTML files to process
html_files = [
    FOLDER / "feminine wipes _ Walgreens one.html", # Updated filename based on user request
    FOLDER / "feminine wipes _ Walgreens two.html" # Updated filename based on user request
]

def clean(x):
    """Cleans up whitespace from a string."""
    return re.sub(r"\s+", " ", x or "").strip()

def first_text(node, selectors):
    """Finds the first matching element by selectors and returns its cleaned text."""
    for sel in selectors:
        el = node.select_one(sel)
        if el:
            txt = clean(el.get_text(" ", strip=True))
            if txt:
                return txt
    return ""

def first_attr(node, selectors, attr):
    """Finds the first matching element by selectors and returns the value of its attribute."""
    for sel in selectors:
        el = node.select_one(sel)
        if el and el.has_attr(attr):
            val = clean(el.get(attr))
            if val:
                return val
    return ""

def extract_rating_count(text):
    """Extracts rating score and count from a text string."""
    if not text:
        return "", ""
    t = clean(text)
    rating = ""
    count = ""
    # Pattern for rating: e.g., "4.5 out of 5", "4.5"
    m_rating = re.search(r"([0-5](?:\.\d)?)\s*(?:out of 5)?", t, re.I)
    if m_rating:
        rating = m_rating.group(1)
    # Pattern for count: e.g., "1,234 ratings", "500 reviews"
    m_count = re.search(r"([\d,]+)\s*(?:ratings?|reviews?)", t, re.I)
    if m_count:
        count = m_count.group(1).replace(",", "")
    if not count:
        # Fallback: if no specific "ratings/reviews" found, try to find any numbers
        nums = re.findall(r"[\d,]+", t)
        if nums:
            count = nums[-1].replace(",", "") # Assume the last number is the count

    return rating, count

def parse_walgreens_item(tile_soup) -> dict | None:
    """
    Extracts ProductWebLink, ProductTitle, UnitPrice, PricePer100ct, Rating, RatingCount
    from a single Walgreens product tile.
    """

    link = first_attr(tile_soup, [
        'a[data-qa="product_link"]',
        'a[href*="/store/"]' # More specific link to a product page
    ], "href")

    # Title extraction, ordered by specificity and common patterns on product detail pages
    title = first_text(tile_soup, [
        'h1[itemprop="name"]', # Most common for product name on detail pages
        '#product-details h1.product-details__product-name', # Specific for product details page
        'h2.product-card__title',
        'a.product-card__title',
        '.wag-product-card__product-name',
        'p.wag-product-card__product-name',
        'span[data-qa="product_name"]',
        'a[data-automation="product-name"]',
        '.product-name-link',
        '.product-card__details__name',
        'h3.wag-product-card__product-name',
        'span[itemprop="name"]',
        'div[itemprop="name"]',
        'h2',
        'h3'
    ])
    if not title: # Fallback to aria-label on link or alt text of image
        title = first_attr(tile_soup, [
            'a.product-card__link',
            'a[href*="/store/"]',
            'img.product-card__image[alt]'
        ], "aria-label")
    if not title:
        title = first_attr(tile_soup, [
            'img.product-card__image'
        ], "alt")
    if not title:
        title = first_text(tile_soup, [
            'p.ProductTitle_ProductTitle__name__24G1w'
        ])

    # Product Price extraction, ordered by specificity
    price = first_text(tile_soup, [
        'div[itemprop="price"]',
        'span[itemprop="price"]',
        '#product-details span.wag-price-display__amount', # Specific for product detail page
        '.wag-price-display__actual-value',
        'span[data-automation="product-price"]',
        '.product-card__price',
        '.product__price',
        'span[data-qa="current_price"]',
        'span.wag-price-display__amount',
        '.price',
        'span.current-price',
        'div.price-value',
        'strong.price',
        '#product-details .wag-price-display__actual-value' # Another specific for product details
    ])
    # Refined regex for price: allows for optional '$' and captures digits, comma, decimal.
    m = re.search(r"\$?\s*([\d,]+\.\d{2})", price)
    unit_price = m.group(1) if m else ""

    # Price / sheet or Price/ea (default to "0" if not found)
    price_per_unit_measurement = "0"
    all_text = clean(tile_soup.get_text(" ", strip=True))

    # Try finding "price/unit" pattern, e.g., "$X.XX/oz", "$X.XX/ea", "$X.XX/sheet", "$X.XX/ct", "$X.XX/item"
    m_unit_price = re.search(r"(\$[\d,]+(?:\.\d{1,2})?)\s*/\s*(?:oz|ea|ct|item|sheet)\b", all_text, re.I)
    if m_unit_price:
        price_per_unit_measurement = m_unit_price.group(0).replace(" ", "")

    # Rating and Rating Count extraction
    rating_text = first_attr(tile_soup, [
        'div[itemprop="aggregateRating"][aria-label]',
        '.wag-rating-stars[aria-label]',
        '.wag-review-stars[aria-label]',
        '[data-qa="rating_stars_aria_label"]',
        'div.product-rating[aria-label]', # Specific to some detail pages
        'div.BVRRRatingSummaryLink, div.product-details__rating-summary' # Specific for product detail page rating summary
    ], "aria-label")

    if not rating_text:
        rating_text = first_text(tile_soup, [
            'span[itemprop="ratingValue"]',
            'span[itemprop="ratingCount"]',
            '.wag-review-stars__average',
            '.wag-review-stars__reviews-count',
            'span.product-card__rating-score',
            '.wag-rating-stars__score',
            '.wag-review-stars__rating-summary',
            'span[data-qa="rating_value"]',
            'span[data-qa="reviews_count"]',
            '.wag-reviews-count__label',
            'div.review-total', # A common container for rating text
            'div.BVRRRatingSummaryLink, div.product-details__rating-summary' # Specific for product detail page rating summary
        ])

    rating, rating_counts = extract_rating_count(rating_text)

    if link or title:
        return {
            "product_web_link": urljoin(BASE, link) if link else "",
            "product_title": title,
            "product_unit_price": unit_price,
            "price_per_100_ct": price_per_unit_measurement, # This field will now contain the 'price / sheet or price/ ea'
            "rating": rating,
            "rating_counts": rating_counts,
            "source_file": None,
        }
    return None

all_results = []

for file_path in html_files:
    print(f"Processing file: {file_path.name}")
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            soup = BeautifulSoup(f, "lxml") # Using lxml for speed if available

        # Select all product tiles for Walgreens
        # Given these are likely product detail pages, the 'product_tiles' selector might find
        # just one main product content area, which is expected.
        product_tiles = soup.select(
            'div.product-card-container, li.wag-product-card, div[data-shelf-item-id], [data-qa="product_tile"], .wag-product-card-wrapper, div[itemtype*="Product"], div[class*="product-details-page"], div#product-details-page' # Added #product-details-page
        )
        # If no specific product cards are found (e.g. on a product detail page),
        # assume the entire soup represents the product details.
        if not product_tiles:
            product_tiles = [soup] # Treat the whole document as the 'tile'

        page_new_count = 0
        seen_on_page = set() # To handle potential duplicates within a single HTML page

        for tile in product_tiles:
            # Use a unique identifier for the tile within the page, e.g., its HTML representation
            tile_key = str(tile)[:500] # Use a portion of the HTML as a key
            if tile_key in seen_on_page:
                continue
            seen_on_page.add(tile_key)

            data = parse_walgreens_item(tile) # Call the new Walgreens-specific parser
            if data:
                data["source_file"] = file_path.name # Add source file info
                all_results.append(data)
                page_new_count += 1
        print(f"  Parsed {page_new_count} products from {file_path.name}.")
    except FileNotFoundError:
        print(f"  Error: File not found - {file_path.name}. Skipping.")
    except Exception as e:
        print(f"  Error processing {file_path.name}: {e}. Skipping.")

if all_results:
    # Create DataFrame and drop duplicates based on product_web_link and product_title
    df = pd.DataFrame(all_results).drop_duplicates(subset=["product_web_link", "product_title"]) # Ensure the columns exist after parsing

    output_csv_path = FOLDER / "walgreens_feminine_wipes_extracted.csv" # Output to the same folder as input files

    # Define fieldnames explicitly to control CSV column order
    fieldnames = [
        "product_web_link",
        "product_title",
        "product_unit_price",
        "price_per_100_ct", # This field will now contain the 'price / sheet or price/ ea'
        "rating",
        "rating_counts",
        "source_file",
    ]
    with open(output_csv_path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        # Convert DataFrame rows to dictionary records for DictWriter
        writer.writerows(df.to_dict('records'))

    print(f"Done! Saved {len(df)} unique items to {output_csv_path}")
    print(df.head(10)) # Display head of the final DataFrame
else:
    print("No results collected from any files; nothing written.")

Processing file: feminine wipes _ Walgreens one.html
  Parsed 1 products from feminine wipes _ Walgreens one.html.
Processing file: feminine wipes _ Walgreens two.html
  Parsed 1 products from feminine wipes _ Walgreens two.html.
Done! Saved 2 unique items to /content/drive/My Drive/Personal Care Project/Web Scraping_Wipes/Walgreens Feminine Wipes/walgreens_feminine_wipes_extracted.csv
                                    product_web_link product_title  \
0  https://www.walgreens.com/store/c/cottonelle-s...                 
1  https://www.walgreens.com/store/c/always-pads-...                 

  product_unit_price price_per_100_ct rating rating_counts  \
0                            $0.25/ea                        
1                            $0.49/ea                        

                           source_file  
0  feminine wipes _ Walgreens one.html  
1  feminine wipes _ Walgreens two.html  
